In [ ]:
# =========================
# Imports
# =========================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split

#fill the code(1) (we need matplotlib)
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns




In [ ]:
# =========================
# Device
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# =========================
# Dataset Path
# =========================

data_dir = "Chessman-image-dataset/Chess"



In [ ]:
# =========================
# Transforms
# =========================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    #fill the code(2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

#hint from Hanan: برای پر کردن کد به عملیات ترنسفرم داده ی آموزش نگاه کنید. کدوم عملیات فقط روی داده ی آموزشی انجام میشه و کدوم روی همه ی داده ها؟


In [ ]:
# =========================
# Load Dataset
# =========================

base_dataset = datasets.ImageFolder(root=data_dir)

train_full = datasets.ImageFolder(
    root=#fill the code(3),
    transform=train_transform
)

eval_full = datasets.ImageFolder(
    root=data_dir,
    transform=#fill the code(4)
)

class_names = base_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)
print("Total images:", len(base_dataset))


# =========================
# Train / Validation / Test Split
# =========================

train_size = int(0.7 * len(base_dataset))
val_size = int(0.15 * len(base_dataset))
test_size = len(base_dataset) #fill the code (5)

#hint from Hanan: به عملیات دقت کنید. ما بخشی از داده رو برای آموزش و اعتبار سنجی برای آموزشی برداشتیم. پس اون چیزی که باقی میمونه رو برای تست برمیداریم.

generator = torch.Generator().manual_seed(42)

train_indices, val_indices, test_indices = random_split(
    range(len(base_dataset)),
    [train_size, val_size, test_size],
    generator=generator
)

train_dataset = Subset(train_full, train_indices.indices)
val_dataset = Subset(eval_full, #fill the code(6))
test_dataset = Subset(#fill the code(7))
    
#hint from Hanan: این قسمت رو عجله نکنید و دقت کنید. ما دو مدل عملیات ترنسفرم تعریف کردیم. یکی برای داده ی آموزش و یکی برای ولیدیشن و تست

print("Train images:", len(train_dataset))
print(#fill the code (8))
print("Test images:", #fill the code(9))


# =========================
# DataLoaders
# =========================

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = #fill the code (10)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=#fill the code (11)
)


# =========================
# Show Sample Images
# =========================

images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 6))

for i in range(8):
    image = images[i].permute(1, 2, 0)
    label = labels[i].item()

    plt.subplot(2, 4, i + 1)
    plt.imshow(image)
    plt.title(class_names[label])
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# =========================
# Model
# =========================



from torchvision import models

model = models.resnet18(pretrained=True)

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)

model = model.to(device)

print(model)

# =========================
# Loss and Optimizer
# =========================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    #fill the code(12)
)

#hint from Hanan: ما اینجا به لرنینگ ریت نیاز داریم. روی عدد لرنینگ ریت سختگیری نمیکنیم. ولی عددای مختلف رو امتحان کنید. 

# =========================
# Training Loop
# =========================

num_epochs = 15 # try different numbers here

train_losses = []
#fill the code (13)

#fill the code (14)
val_accuracies = []

for epoch in range(num_epochs):

    # ---- Training ----
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        #fill the code (15)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = #fill the code (16)

    train_losses.append(#fill the code (17))
    train_accuracies.append(train_acc)
        
    #hint from Hanan = میخوایم مقدار دقت و لاس رو حساب کنیم. باید فرمول ها رو تکمیل کنید.

    # ---- Validation ----
    model.eval()

    val_running_loss = 0.0
    #fill the code (18)
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = #fill the code (19)
    val_acc = val_correct / val_total

    val_losses.append(val_loss)
    #fill the code (20)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f}"
    )



In [ ]:
# =========================
# Plot Loss Curves
# =========================

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()


# =========================
# Plot Accuracy Curves
# =========================

plt.figure(figsize=(8, 5))
plt.plot(train_accuracies, label="Training Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()

# just run this cell :D

In [ ]:
# =========================
# Final Test Evaluation
# =========================

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


# =========================
# Test Accuracy
# =========================

correct = 0

for pred, label in zip(all_preds, all_labels):
    if pred == label:
        correct += 1

test_accuracy = correct / #fill the code (21)

print("Test Accuracy:", #fill the code (22))


# =========================
# Classification Report
# =========================

print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names
))




# =========================
# Show Some Test Predictions
# =========================

images, labels = next(iter(test_loader))

model.eval()

with torch.no_grad():
    outputs = model(images.to(device))
    _, predicted = torch.max(outputs, 1)

plt.figure(figsize=(10, 6))

for i in range(8):
    image = images[i].permute(1, 2, 0)

    true_label = class_names[labels[i].item()]
    pred_label = class_names[predicted[i].item()]

    plt.subplot(2, 4, i + 1)
    plt.imshow(image)
    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis("off")

plt.tight_layout()
plt.show()